# GitHub Issue Analyzer

## First-time setup
Run the setup cell below **once**. It performs a 10-step provisioning:
1. IAM permissions for the execution role
2. DynamoDB tables (IssueClassifications, RecommendationResults)
3. AgentCore Memory resource
4. Lambda execution IAM role
5. Gateway IAM role
6. Lambda function deployment (tool handlers)
7. Cognito User Pool + App Client + test user
8. AgentCore Gateway with CUSTOM_JWT auth
9. Gateway Target pointing at the Lambda function
10. AgentCore Runtime execution IAM role

In [ ]:
%pip install -r requirements.txt -q

In [ ]:
!python setup_aws.py

## Launch the app
Run the cells below to get the URL and start Streamlit.

In [ ]:
import json
import boto3

try:
    with open('/opt/ml/metadata/resource-metadata.json', 'r') as f:
        data = json.load(f)
    domain_id = data['DomainId']
    space_name = data['SpaceName']
    sagemaker_client = boto3.client('sagemaker')
    response = sagemaker_client.describe_space(DomainId=domain_id, SpaceName=space_name)
    url = response['Url'] + '/proxy/8501/'
except Exception:
    url = 'http://localhost:8501'

print(f'Access the app at:\n{url}')

In [ ]:
!streamlit run app.py

## Deploy Recommender Agent to AgentCore Runtime

The cells below containerize the Recommender Agent, push to ECR via CodeBuild, and deploy to AgentCore Runtime with full observability.

**Prerequisites:** Run the setup cells above first (steps 1-10 must complete successfully).

In [ ]:
import boto3
from bedrock_agentcore_starter_toolkit import Runtime

ssm = boto3.client("ssm", region_name="us-west-2")

# Read required config from SSM
execution_role_arn = ssm.get_parameter(
    Name="/app/issueanalyzer/agentcore/runtime_execution_role_arn", WithDecryption=True
)["Parameter"]["Value"]
cognito_client_id = ssm.get_parameter(
    Name="/app/issueanalyzer/agentcore/cognito_client_id", WithDecryption=True
)["Parameter"]["Value"]
cognito_discovery_url = ssm.get_parameter(
    Name="/app/issueanalyzer/agentcore/cognito_discovery_url", WithDecryption=True
)["Parameter"]["Value"]

print(f"Execution Role: {execution_role_arn}")
print(f"Cognito Client ID: {cognito_client_id}")
print(f"Discovery URL: {cognito_discovery_url}")

# Configure runtime
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="runtime_agent.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region="us-west-2",
    agent_name="issue_analyzer_recommender",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_client_id],
            "discoveryUrl": cognito_discovery_url,
        }
    },
    request_header_configuration={
        "requestHeaderAllowlist": [
            "Authorization",
        ]
    },
)

print("Runtime configured successfully!")
print(f"Config path: {response}")

### Launch Runtime
This cell builds the Docker image via CodeBuild, pushes to ECR, and deploys to AgentCore Runtime. **This takes several minutes.**

In [ ]:
memory_id = ssm.get_parameter(
    Name="/app/issueanalyzer/agentcore/memory_id", WithDecryption=True
)["Parameter"]["Value"]

print(f"Launching with MEMORY_ID={memory_id}")

launch_result = agentcore_runtime.launch(
    env_vars={"MEMORY_ID": memory_id},
    auto_update_on_conflict=True,
)

print(f"Agent ARN: {launch_result.agent_arn}")

# Save ARN to SSM
ssm.put_parameter(
    Name="/app/issueanalyzer/agentcore/runtime_arn",
    Value=launch_result.agent_arn,
    Type="String",
    Overwrite=True,
)
print(f"Saved runtime ARN to SSM")

### Poll Runtime Status
Wait until the runtime status becomes **READY**.

In [ ]:
import time

end_statuses = {"READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"}

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
print(f"Current status: {status}")

while status not in end_statuses:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(f"Status: {status}")

if status == "READY":
    print("\nRuntime is READY!")
else:
    print(f"\nRuntime ended with status: {status}")

### Test Runtime Invocation
Send a test request to the deployed Recommender Agent.

In [ ]:
import uuid
import hashlib
import hmac
import base64

# Get a fresh Cognito access token
cognito_client_secret = ssm.get_parameter(
    Name="/app/issueanalyzer/agentcore/cognito_client_secret", WithDecryption=True
)["Parameter"]["Value"]

cognito = boto3.client("cognito-idp", region_name="us-west-2")
username = "agent_user"
password = "AgentPass123!"

message = bytes(username + cognito_client_id, "utf-8")
key = bytes(cognito_client_secret, "utf-8")
secret_hash = base64.b64encode(
    hmac.new(key, message, digestmod=hashlib.sha256).digest()
).decode()

auth_response = cognito.initiate_auth(
    ClientId=cognito_client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={
        "USERNAME": username,
        "PASSWORD": password,
        "SECRET_HASH": secret_hash,
    },
)
access_token = auth_response["AuthenticationResult"]["AccessToken"]
print("Cognito authentication successful")

# Invoke the runtime
session_id = str(uuid.uuid4())
response = agentcore_runtime.invoke(
    payload={
        "prompt": "Analyze the classified issues for the most recently analyzed repository. Read the classification data and return your analysis as a JSON object.",
        "actor_id": "notebook_test_user",
    },
    bearer_token=access_token,
    session_id=session_id,
)

print(f"\nRuntime Response:\n{response.get('response', response)}")

### Set Up Online Evaluations
Create an evaluation configuration that monitors agent quality using three built-in evaluators.

In [ ]:
import json
from bedrock_agentcore_starter_toolkit import Evaluation
from pathlib import Path

eval_client = Evaluation(region="us-west-2")

# Get agent ID from the runtime ARN
agent_arn = ssm.get_parameter(
    Name="/app/issueanalyzer/agentcore/runtime_arn", WithDecryption=True
)["Parameter"]["Value"]
agent_id = agent_arn.split(":")[-1].split("/")[-1]

print(f"Agent ID: {agent_id}")

# Point the runtime client config at the yaml generated during configure()
agentcore_runtime._config_path = Path.cwd() / ".bedrock_agentcore.yaml"

# Create online evaluation config
eval_response = eval_client.create_online_config(
    agent_id=agent_id,
    config_name="issue_analyzer_eval",
    sampling_rate=100,
    evaluator_list=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolSelectionAccuracy",
    ],
    config_description="Issue Analyzer Recommender Agent online evaluation",
    auto_create_execution_role=True,
)

eval_config_id = eval_response["onlineEvaluationConfigId"]
print(f"Evaluation Config ID: {eval_config_id}")

# Verify
config_details = eval_client.get_online_config(config_id=eval_config_id)
print(json.dumps(config_details, indent=2, default=str))

### Observability
Traces and logs are automatically available in CloudWatch:
- **GenAI Observability Dashboard:** CloudWatch > GenAI Observability > Bedrock AgentCore
- **Agent Logs:** `/aws/bedrock-agentcore/runtimes/<agent-id>-DEFAULT`
- **Evaluation Results:** `/aws/bedrock-agentcore/evaluations/results/<config-id>`